# Mini Project 3: Feature Engineering with S3, Lambda, and DynamoDB

This notebook provides a Free Tier friendly scaffold for building a lightweight feature engineering workflow that reads raw events from S3, computes reusable features inside Lambda, stores feature versions back to S3, and records metadata inside DynamoDB. Use it as a starting point before promoting the code to managed CI/CD tooling.


## Architecture & Prerequisites
- **Data plane**: Raw data lands in `s3://<raw-bucket>/raw/...`; engineered features persist to `s3://<feature-bucket>/features/<feature-set>/v_<timestamp>.csv`.
- **Compute plane**: A Python 3.11 Lambda function applies transformations, writes outputs, and updates DynamoDB metadata.
- **Metadata plane**: DynamoDB table `feature_catalog` stores feature schemas, owners, quality metrics, and run history.
- **Scheduling**: CloudWatch Events (EventBridge) triggers the Lambda on a cron or rate schedule; ad-hoc backfills can be driven locally with the helper functions below.
- **Before running**: Configure IAM roles with S3 + DynamoDB + CloudWatch permissions, set AWS credentials locally, and keep datasets <= 5 GB to remain in the Free Tier.


In [1]:

import io
import json
import logging
import os
import uuid
from datetime import datetime, timedelta
from decimal import Decimal
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError

# Environment-driven configuration keeps the notebook portable across accounts.
AWS_REGION = os.getenv('AWS_REGION', 'us-east-1')
RAW_BUCKET = os.getenv('RAW_FEATURE_BUCKET', 'mlu-free-tier-raw')
FEATURE_BUCKET = os.getenv('FEATURE_STORE_BUCKET', 'mlu-free-tier-feature-store')
RAW_PREFIX = os.getenv('RAW_PREFIX', 'raw_transactions')
FEATURE_PREFIX = os.getenv('FEATURE_PREFIX', 'features')
FEATURE_TABLE = os.getenv('FEATURE_TABLE', 'feature_catalog')
FEATURE_SET_NAME = os.getenv('FEATURE_SET', 'customer_spending_features')

session = boto3.Session(region_name=AWS_REGION)
s3_client = session.client('s3')
dynamodb = session.resource('dynamodb')
cloudwatch_events = session.client('events')

logger = logging.getLogger('feature-store')
logger.setLevel(logging.INFO)

print(f"Region: {AWS_REGION}
Raw bucket: {RAW_BUCKET}
Feature bucket: {FEATURE_BUCKET}
Feature table: {FEATURE_TABLE}")


SyntaxError: unterminated string literal (detected at line 32) (3277322472.py, line 32)

## Step 1. Design Feature Storage in S3
Plan the folder structure before uploading data:

```
s3://<raw-bucket>/raw_transactions/<source>/<date>/batch.json
  Event stream snapshots (JSON/CSV)

s3://<feature-bucket>/features/<feature_set>/v_<YYYYMMDDHHMMSS>.csv
  Engineered feature versions (CSV or Parquet)

s3://<feature-bucket>/metadata/<feature_set>/data_dictionary.json
  Optional documentation + data dictionaries
```

The helper below idempotently creates buckets/prefixes (creating buckets may require the AWS CLI or IaC in hardened accounts).


In [ ]:

def ensure_bucket(client, bucket_name, region):
    '''Best-effort bucket creation. Will no-op if the bucket already exists.'''
    try:
        client.head_bucket(Bucket=bucket_name)
        logger.info('Bucket %s already exists', bucket_name)
        return
    except ClientError as exc:
        error_code = exc.response.get('Error', {}).get('Code', '')
        if error_code not in ('404', 'NoSuchBucket'):
            raise
    params = {'Bucket': bucket_name}
    if region != 'us-east-1':
        params['CreateBucketConfiguration'] = {'LocationConstraint': region}
    client.create_bucket(**params)
    logger.info('Created bucket %s in %s', bucket_name, region)


def ensure_prefix(client, bucket_name, prefix):
    key = prefix.rstrip('/') + '/'
    client.put_object(Bucket=bucket_name, Key=key, Body=b'')
    logger.info('Ensured prefix s3://%s/%s', bucket_name, prefix)


ensure_bucket(s3_client, RAW_BUCKET, AWS_REGION)
ensure_bucket(s3_client, FEATURE_BUCKET, AWS_REGION)
for prefix in (RAW_PREFIX, f"{FEATURE_PREFIX}/{FEATURE_SET_NAME}", f"metadata/{FEATURE_SET_NAME}"):
    target_bucket = FEATURE_BUCKET if prefix.startswith('metadata') or prefix.startswith(FEATURE_PREFIX) else RAW_BUCKET
    ensure_prefix(s3_client, target_bucket, prefix)


## Step 2. Generate Sample Raw Events and Upload to S3
Use lightweight synthetic data (< 1 MB) to iterate locally before wiring real sources. The snippet below creates spending events with categorical + numerical features, writes JSON + CSV variants, and uploads them to the raw bucket for Lambda to consume.


In [ ]:

def build_sample_transactions(num_records: int = 200) -> pd.DataFrame:
    rng = np.random.default_rng(42)
    base_time = datetime.utcnow() - timedelta(hours=4)
    merchants = ['groceries', 'fuel', 'electronics', 'travel', 'subscriptions']
    channels = ['mobile', 'web', 'pos']
    records = []
    for i in range(num_records):
        ts = base_time + timedelta(minutes=i * 3)
        records.append(
            {
                'transaction_id': str(uuid.uuid4()),
                'customer_id': int(rng.integers(10_000, 20_000)),
                'merchant_type': str(rng.choice(merchants)),
                'channel': str(rng.choice(channels, p=[0.5, 0.3, 0.2])),
                'amount': round(float(rng.normal(75, 22)), 2),
                'is_fraud': bool(rng.binomial(1, 0.05)),
                'event_timestamp': ts.isoformat(),
            }
        )
    return pd.DataFrame(records)


def upload_raw_snapshot(df: pd.DataFrame, bucket: str, prefix: str) -> str:
    snapshot_id = datetime.utcnow().strftime('%Y%m%d%H%M%S')
    key_prefix = f"{prefix}/transactions/{snapshot_id}"
    key_json = key_prefix + '.json'
    key_csv = key_prefix + '.csv'
    s3_client.put_object(Bucket=bucket, Key=key_json, Body=df.to_json(orient='records').encode('utf-8'))
    s3_client.put_object(Bucket=bucket, Key=key_csv, Body=df.to_csv(index=False).encode('utf-8'))
    logger.info('Uploaded %s records to s3://%s/%s', len(df), bucket, key_json)
    return key_json


sample_df = build_sample_transactions(250)
latest_key = upload_raw_snapshot(sample_df, RAW_BUCKET, RAW_PREFIX)
latest_key


## Step 3. Lambda Feature Processing
The Lambda handler should: (1) read the raw snapshot indicated by the S3 event, (2) engineer normalized/aggregated features, (3) write a versioned feature file, and (4) update DynamoDB with schema + lineage. The following template keeps dependencies minimal (pure pandas + boto3) so it fits inside the free deployment package limit.


In [ ]:

def fetch_raw_dataframe(bucket: str, key: str) -> pd.DataFrame:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    body = obj['Body'].read()
    if key.endswith('.json'):
        return pd.read_json(io.BytesIO(body))
    return pd.read_csv(io.BytesIO(body))


def build_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    features = df.copy()
    features['amount_zscore'] = (features['amount'] - features['amount'].mean()) / features['amount'].std(ddof=0)
    features['amount_rolling_3'] = features['amount'].rolling(window=3, min_periods=1).mean()
    features['fraud_rolling_rate'] = (
        features['is_fraud'].astype(int).rolling(window=20, min_periods=5).mean().fillna(0)
    )
    features = pd.get_dummies(features, columns=['merchant_type', 'channel'], prefix=['m', 'ch'], drop_first=True)
    features['event_timestamp'] = pd.to_datetime(features['event_timestamp'])
    features.sort_values('event_timestamp', inplace=True)
    return features.reset_index(drop=True)


def store_feature_frame(df: pd.DataFrame, bucket: str, prefix: str, feature_set: str) -> dict:
    version = datetime.utcnow().strftime('%Y%m%d%H%M%S')
    feature_key = f"{prefix}/{feature_set}/v_{version}.csv"
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3_client.put_object(Bucket=bucket, Key=feature_key, Body=buffer.getvalue().encode('utf-8'))
    logger.info('Stored feature set at s3://%s/%s', bucket, feature_key)
    return {'feature_key': feature_key, 'version': version, 'record_count': len(df)}


def df_columns_to_schema(dtypes) -> list[dict]:
    schema = []
    for name, dtype in dtypes.items():
        schema.append({'name': name, 'dtype': str(dtype), 'description': ''})
    return schema


def update_feature_catalog(table_name: str, feature_set: str, metadata: dict) -> None:
    table = dynamodb.Table(table_name)
    item = {
        'feature_set': feature_set,
        'version': metadata['version'],
        's3_key': metadata['feature_key'],
        'record_count': metadata['record_count'],
        'generated_at': datetime.utcnow().isoformat(),
        'schema': metadata['columns'],
        'quality': metadata.get('quality', {}),
    }
    table.put_item(Item=item)
    logger.info('Upserted catalog entry for %s v%s', feature_set, metadata['version'])


def lambda_handler(event, context):
    logger.info('Received event: %s', json.dumps(event))
    for record in event.get('Records', []):
        bucket = record['s3']['bucket']['name']
        key = record['s3']['object']['key']
        df = fetch_raw_dataframe(bucket, key)
        feature_df = build_feature_frame(df)
        metadata = store_feature_frame(feature_df, FEATURE_BUCKET, FEATURE_PREFIX, FEATURE_SET_NAME)
        metadata['columns'] = df_columns_to_schema(feature_df.dtypes)
        metadata['quality'] = {
            'null_ratio': feature_df.isna().mean().round(4).to_dict(),
            'source_object': key,
        }
        update_feature_catalog(FEATURE_TABLE, FEATURE_SET_NAME, metadata)
    return {'statusCode': 200, 'detail': 'Processed feature batches', 'records': len(event.get('Records', []))}


In [ ]:

# Quick local dry-run without touching AWS services.
local_sample = build_sample_transactions(50)
feature_preview = build_feature_frame(local_sample)
feature_preview.head()


## Step 4. DynamoDB Feature Catalog Helpers
The catalog tracks data lineage, schema changes, owners, and quality scores. Define a thin repository layer that the Lambda + local discovery client can share. Storing floats as `Decimal` keeps DynamoDB happy.


In [ ]:

def to_decimal_map(payload: dict) -> dict:
    converted = {}
    for key, value in payload.items():
        if isinstance(value, float):
            converted[key] = Decimal(str(round(value, 6)))
        elif isinstance(value, dict):
            converted[key] = to_decimal_map(value)
        else:
            converted[key] = value
    return converted


def register_feature_definition(feature_set: str, owner: str, description: str, schema: list[dict]):
    table = dynamodb.Table(FEATURE_TABLE)
    item = {
        'feature_set': feature_set,
        'version': 'definition',
        'owner': owner,
        'description': description,
        'schema': schema,
        'updated_at': datetime.utcnow().isoformat(),
    }
    table.put_item(Item=item)
    logger.info('Registered definition for %s', feature_set)


def log_feature_run(feature_set: str, version: str, s3_key: str, quality: dict):
    table = dynamodb.Table(FEATURE_TABLE)
    item = {
        'feature_set': feature_set,
        'version': version,
        's3_key': s3_key,
        'quality': to_decimal_map(quality),
        'created_at': datetime.utcnow().isoformat(),
    }
    table.put_item(Item=item)
    logger.info('Logged feature run %s v%s', feature_set, version)


## Step 5. Local Feature Discovery Interface
Analysts should be able to search for feature sets, inspect schemas, download the latest batch, and join with local notebooks. The class below provides a lightweight client over DynamoDB + S3 for that purpose.


In [ ]:

class FeatureCatalogClient:
    def __init__(self, table_name: str = FEATURE_TABLE, s3_bucket: str = FEATURE_BUCKET, s3_prefix: str = FEATURE_PREFIX):
        self.table = dynamodb.Table(table_name)
        self.bucket = s3_bucket
        self.prefix = s3_prefix

    def list_feature_sets(self):
        response = self.table.scan(ProjectionExpression='feature_set, version, owner, description')
        return response.get('Items', [])

    def latest_version(self, feature_set: str) -> dict:
        response = self.table.query(
            KeyConditionExpression='feature_set = :fs',
            ExpressionAttributeValues={':fs': feature_set},
            ScanIndexForward=False,
            Limit=1,
        )
        items = response.get('Items', [])
        return items[0] if items else {}

    def download_features(self, feature_set: str) -> pd.DataFrame:
        version_info = self.latest_version(feature_set)
        if not version_info:
            raise ValueError(f'No versions found for {feature_set}')
        key = version_info['s3_key']
        obj = s3_client.get_object(Bucket=self.bucket, Key=key)
        return pd.read_csv(io.BytesIO(obj['Body'].read()))


catalog = FeatureCatalogClient()
# catalog.list_feature_sets()


## Step 6. Batch Feature Pipeline & Scheduling
Use CloudWatch Events (EventBridge) to trigger the Lambda periodically without leaving the Free Tier. The sample rule below executes every 15 minutes and targets the Lambda ARN you deploy via SAM/Serverless.


In [ ]:

def create_schedule(rule_name: str, lambda_arn: str, rate_expression: str = 'rate(15 minutes)'):
    cloudwatch_events.put_rule(Name=rule_name, ScheduleExpression=rate_expression, State='ENABLED')
    cloudwatch_events.put_targets(
        Rule=rule_name,
        Targets=[{'Id': 'feature-lambda', 'Arn': lambda_arn}]
    )
    logger.info('Attached schedule %s -> %s', rule_name, lambda_arn)


# Example usage (uncomment after deploying Lambda):
# create_schedule('feature-refresh-quarter-hour', 'arn:aws:lambda:us-east-1:123456789012:function:feature-transform')


## Step 7. Next Steps & Validation Checklist
- [ ] Package the Lambda with this handler (SAM or AWS CLI `zip`) and deploy with an IAM role granting S3 + DynamoDB + CloudWatch access.
- [ ] Create the DynamoDB table (`feature_catalog`) with primary key (`feature_set`, `version`).
- [ ] Run the notebook locally to push a seed feature batch and verify catalog entries.
- [ ] Configure EventBridge schedule + DLQ/alarms for failure handling.
- [ ] Add unit tests (e.g., `pytest`) for `build_feature_frame` and catalog helpers before promoting to production.
